# FahMai RAG v5 — Pinecone + OpenRouter Embeddings + Typhoon API

Same pipeline as v5 but with:
- **Embeddings**: `text-embedding-3-large` via OpenRouter (dim=3072)
- **Vector DB**: Pinecone (replaces in-memory numpy matrix)
- **Hybrid search**: Pinecone dense + BM25 → RRF merge
- **LLM**: Typhoon API (thaillm.or.th)

Required env vars / `.env` file:
```
OPENROUTER_API_KEY=sk-or-...
PINECONE_API_KEY=pcsk_...
THAILLM_API_KEY=...
```

In [ ]:
!unzip /content/super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip

Archive:  /content/super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip
  inflating: data/knowledge_base/policies/cancellation_policy.md  
  inflating: data/knowledge_base/policies/membership_points_policy.md  
  inflating: data/knowledge_base/policies/return_policy.md  
  inflating: data/knowledge_base/policies/shipping_policy.md  
  inflating: data/knowledge_base/policies/warranty_policy.md  
  inflating: data/knowledge_base/products/AW-MN-001_arcwave_proview_27_4k.md  
  inflating: data/knowledge_base/products/AW-SK-001_arcwave_soundpillar_300.md  
  inflating: data/knowledge_base/products/DN-DT-001_daonuea_tower_x10.md  
  inflating: data/knowledge_base/products/DN-DT-002_daonuea_tower_x10_max.md  
  inflating: data/knowledge_base/products/DN-DT-003_daonuea_mini_pc_m1.md  
  inflating: data/knowledge_base/products/DN-DT-004_daonuea_all_in_one_27.md  
  inflating: data/knowledge_base/products/DN-DT-005_daonuea_all_in_one_24.md  
  inflating: data/knowledge_base/products/DN-LT-001

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
!pip install -q pinecone openai rank-bm25 pythainlp requests python-dotenv pandas \
    sentence-transformers tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 17.6 MB/s eta 0:00:00


In [ ]:
import os, csv, re, time, json, requests, random
import numpy as np
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()  # reads .env in working directory

# ── Config ──────────────────────────────────────────────────────────────────
N_QUESTIONS       = 100
DATA_DIR          = './data'  # adjust if needed
KB_DIR            = f'{DATA_DIR}/knowledge_base'

N_VOTES           = 1     # shuffled LLM calls per question
N_QUERIES         = 5
TOP_K_RETRIEVE    = 40
TOP_K_RERANK      = 15

EMBED_MODEL       = 'openai/text-embedding-3-large'
EMBED_DIM         = 3072
EMBED_BATCH       = 64    # texts per OpenRouter embedding request

PINECONE_INDEX    = 'fahmai-rag-v5'
PINECONE_NS       = 'chunks'

OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', '')
PINECONE_API_KEY   = os.environ.get('PINECONE_API_KEY', '')
THAILLM_API_KEY    = os.environ.get('THAILLM_API_KEY', '')

assert OPENROUTER_API_KEY, 'Set OPENROUTER_API_KEY in .env'
assert PINECONE_API_KEY,   'Set PINECONE_API_KEY in .env'
assert THAILLM_API_KEY,    'Set THAILLM_API_KEY in .env'
print('Config ready')

Config ready


---
## Section 1: Load Knowledge Base

In [ ]:
kb_dir = Path(KB_DIR)
documents = []
for fp in sorted(kb_dir.rglob('*.md')):
    text = fp.read_text(encoding='utf-8')
    rel  = fp.relative_to(kb_dir)
    documents.append({
        'path':     str(rel),
        'filename': fp.name,
        'category': rel.parts[0],
        'text':     text
    })
print(f'Loaded {len(documents)} documents')

Loaded 118 documents


---
## Section 2: Chunking (sentence-aware, same as v5)

In [ ]:
def extract_doc_title(text):
    m = re.search(r'^#\s+(.+)', text, re.MULTILINE)
    return m.group(1).strip() if m else ''

def extract_sku(filename):
    return filename.split('_')[0].upper() if '_' in filename else ''


def split_sentence_aware(text, max_chars=600, overlap_chars=100):
    sentence_ends = re.compile(r'(?<=[.!?\n])\s+|(?<=\n)(?=#+\s)')
    sentences = sentence_ends.split(text)
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks, current, overlap_buf = [], '', ''
    for sent in sentences:
        candidate = (overlap_buf + ' ' + sent).strip() if overlap_buf else sent
        if len(current) + len(candidate) + 1 <= max_chars:
            current = (current + ' ' + candidate).strip()
        else:
            if current:
                chunks.append(current)
                overlap_buf = current[-overlap_chars:] if len(current) > overlap_chars else current
            current = candidate
    if current:
        chunks.append(current)
    return chunks if chunks else [text]


def split_by_sections(text, max_chars=600):
    pattern = re.compile(r'(?=^#{1,3}\s)', re.MULTILINE)
    sections = [s.strip() for s in pattern.split(text) if s.strip()]
    chunks = []
    for sec in sections:
        if len(sec) <= max_chars:
            chunks.append(sec)
        else:
            chunks.extend(split_sentence_aware(sec, max_chars=max_chars))
    return chunks


def build_metadata_header(doc, sku=''):
    parts = [f"[หมวด: {doc['category']}]"]
    if doc.get('title'):
        parts.append(f"[สินค้า: {doc['title']}]")
    if sku:
        parts.append(f'[SKU: {sku}]')
    parts.append(f"[ไฟล์: {doc['path']}]")
    return '  '.join(parts)


all_chunks = []
for doc in documents:
    doc['title'] = extract_doc_title(doc['text'])
    sku = extract_sku(doc['filename'])
    header = build_metadata_header(doc, sku)

    for sec in split_by_sections(doc['text'], max_chars=600):
        all_chunks.append({
            'text':     header + '\n' + sec,
            'source':   doc['path'],
            'category': doc['category'],
            'title':    doc['title'],
            'sku':      sku,
            'type':     'section',
        })

    price_lines = re.findall(r'(?:ราคา|price)[^\n]{0,100}', doc['text'], re.IGNORECASE)
    spec_lines  = re.findall(r'(?:RAM|ROM|CPU|GPU|แบต|battery|bluetooth|wifi|ANC|LDAC|GPS|ATM|IP6)[^\n]{0,120}', doc['text'], re.IGNORECASE)
    if price_lines or spec_lines:
        summary = header + '\n[สรุปสเปคและราคา]\n'
        summary += '\n'.join(price_lines[:5] + spec_lines[:8])
        all_chunks.append({
            'text':     summary,
            'source':   doc['path'],
            'category': doc['category'],
            'title':    doc['title'],
            'sku':      sku,
            'type':     'summary',
        })

chunks = all_chunks
n_section = sum(1 for c in chunks if c['type'] == 'section')
n_summary = sum(1 for c in chunks if c['type'] == 'summary')
print(f'Created {len(chunks)} chunks total  (section={n_section}, summary={n_summary})')

Created 1360 chunks total  (section=1242, summary=118)


---
## Section 3: Embed with text-embedding-3-large via OpenRouter

In [ ]:
def embed_texts(texts: list[str], batch_size: int = EMBED_BATCH) -> list[list[float]]:
    """Call OpenRouter embeddings API, return list of float vectors."""
    url     = 'https://openrouter.ai/api/v1/embeddings'
    headers = {
        'Authorization': f'Bearer {OPENROUTER_API_KEY}',
        'Content-Type':  'application/json',
    }
    all_vecs = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Embedding'):
        batch = texts[i : i + batch_size]
        payload = {'model': EMBED_MODEL, 'input': batch}
        for attempt in range(5):
            try:
                resp = requests.post(url, headers=headers, json=payload, timeout=120)
                if resp.status_code == 429:
                    wait = min(2**attempt, 60)
                    print(f'  rate limit, wait {wait}s...')
                    time.sleep(wait); continue
                resp.raise_for_status()
                data = resp.json()['data']
                # data is sorted by index
                vecs = [
                    [float(v) for v in item['embedding']]
                    for item in sorted(data, key=lambda x: x['index'])
                    ]
                all_vecs.extend(vecs)
                break
            except Exception as e:
                wait = min(2**attempt, 30)
                print(f'  embed error: {e}, retry {wait}s...')
                time.sleep(wait)
    return all_vecs


def embed_query(query: str) -> list[float]:
    return embed_texts([query], batch_size=1)[0]


print('embed_texts() ready')

embed_texts() ready


---
## Section 4: Pinecone — Create Index & Upsert Chunks

In [ ]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

# Create index if it doesn't exist
existing = [idx.name for idx in pc.list_indexes()]
if PINECONE_INDEX not in existing:
    print(f'Creating Pinecone index "{PINECONE_INDEX}" dim={EMBED_DIM}...')
    pc.create_index(
        name=PINECONE_INDEX,
        dimension=EMBED_DIM,
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1'),
    )
    # Wait until ready
    while not pc.describe_index(PINECONE_INDEX).status['ready']:
        time.sleep(2)
    print('Index created')
else:
    print(f'Index "{PINECONE_INDEX}" already exists')

index = pc.Index(PINECONE_INDEX)
print(index.describe_index_stats())

Index "fahmai-rag-v5" already exists
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '151',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 28 Mar 2026 21:17:39 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '46',
                                    'x-pinecone-request-latency-ms': '46',
                                    'x-pinecone-response-duration-ms': '47'}},
 'dimension': 3072,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [ ]:
# ── Upsert all chunks (skip if already populated) ─────────────────────────────
stats = index.describe_index_stats()
ns_count = stats.namespaces.get(PINECONE_NS, {}).get('vector_count', 0)

if ns_count >= len(chunks):
    print(f'Index already has {ns_count} vectors, skipping upsert.')
else:
    print(f'Embedding {len(chunks)} chunks...')
    chunk_texts = [c['text'] for c in chunks]
    embeddings  = embed_texts(chunk_texts)

    print('Upserting to Pinecone...')
    UPSERT_BATCH = 100
    for i in tqdm(range(0, len(chunks), UPSERT_BATCH), desc='Upserting'):
        batch_chunks = chunks[i : i + UPSERT_BATCH]
        batch_vecs   = embeddings[i : i + UPSERT_BATCH]
        vectors = [
            {
                'id': f'chunk_{i+j}',
                'values': batch_vecs[j],
                'metadata': {
                    'text':     batch_chunks[j]['text'][:1000],  # Pinecone metadata limit
                    'source':   batch_chunks[j]['source'],
                    'category': batch_chunks[j]['category'],
                    'title':    batch_chunks[j]['title'],
                    'sku':      batch_chunks[j]['sku'],
                    'type':     batch_chunks[j]['type'],
                    'chunk_idx': i + j,
                }
            }
            for j, _ in enumerate(batch_chunks)
        ]
        index.upsert(vectors=vectors, namespace=PINECONE_NS)

    print(f'Upserted {len(chunks)} vectors')
    print(index.describe_index_stats())

Embedding 1360 chunks...


Embedding:   0%|          | 0/22 [00:00<?, ?it/s]

Upserting to Pinecone...


Upserting:   0%|          | 0/14 [00:00<?, ?it/s]

Upserted 1360 vectors
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '183',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 28 Mar 2026 21:18:41 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '47',
                                    'x-pinecone-request-latency-ms': '41',
                                    'x-pinecone-response-duration-ms': '48'}},
 'dimension': 3072,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'chunks': {'vector_count': 1360}},
 'storageFullness': 0.0,
 'total_vector_count': 1360,
 'vector_type': 'dense'}


---
## Section 5: BM25 Index (same as v5)

In [ ]:
from rank_bm25 import BM25Okapi
from pythainlp.tokenize import word_tokenize

def tokenize_thai(text):
    return [t for t in word_tokenize(text.lower(), engine='newmm') if t.strip()]

print('Building BM25 index...')
tokenized_corpus = [tokenize_thai(c['text']) for c in chunks]
bm25_full = BM25Okapi(tokenized_corpus)

summary_idxs   = [i for i, c in enumerate(chunks) if c['type'] == 'summary']
summary_corpus = [tokenized_corpus[i] for i in summary_idxs]
bm25_summary   = BM25Okapi(summary_corpus) if summary_corpus else None

print(f'BM25 full: {len(tokenized_corpus)} docs')
print(f'BM25 summary: {len(summary_corpus)} docs')

Building BM25 index...
BM25 full: 1360 docs
BM25 summary: 118 docs


---
## Section 6: Hybrid Retrieval — Pinecone Dense + BM25 + Multi-query

In [ ]:
def pinecone_retrieve(query: str, top_k: int) -> list[tuple[int, float]]:
    """Embed query → search Pinecone → return list of (chunk_idx, score)."""
    q_vec = embed_query(query)
    res   = index.query(
        vector=q_vec,
        top_k=top_k,
        namespace=PINECONE_NS,
        include_metadata=False,
    )
    results = []
    for match in res['matches']:
        chunk_idx = int(match['id'].split('_')[1])
        results.append((chunk_idx, float(match['score'])))
    return results


def bm25_retrieve(query: str, top_k: int) -> list[tuple[int, float]]:
    """Hybrid BM25: full corpus + boosted summary index."""
    tokens = tokenize_thai(query)
    full_scores = bm25_full.get_scores(tokens)

    if bm25_summary and summary_idxs:
        sum_scores = bm25_summary.get_scores(tokens)
        for local_i, global_i in enumerate(summary_idxs):
            full_scores[global_i] += 1.5 * sum_scores[local_i]

    top_idxs = np.argsort(full_scores)[::-1][:top_k]
    return [(int(i), float(full_scores[i])) for i in top_idxs]


def rrf_merge(ranked_lists, top_k, k=60):
    """Reciprocal Rank Fusion over multiple ranked lists of (idx, score)."""
    rrf_scores = {}
    for ranked in ranked_lists:
        for rank, (idx, _) in enumerate(ranked):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank + 1)
    sorted_idxs = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]
    return [{**chunks[i], 'rrf_score': rrf_scores[i]} for i in sorted_idxs]


def single_query_retrieve(query: str, top_k: int):
    """One query → Pinecone dense + BM25 → RRF merge."""
    dense_r = pinecone_retrieve(query, top_k)
    bm25_r  = bm25_retrieve(query, top_k)
    return rrf_merge([dense_r, bm25_r], top_k)


print('Retrieval functions ready')

Retrieval functions ready


In [ ]:
# ── Multi-query variants (rule-based, same as v5) ─────────────────────────────

PRODUCT_ALIASES = {
    # Watches
    'watch s3 ultra':'wk-sw-001','watch s3 pro':'wk-sw-002',
    'watch s3 se':'wk-sw-004','watch s3':'wk-sw-003',
    'watch s2 pro':'wk-sw-008','band 8 pro':'wk-ft-002','band 8':'wk-ft-001',
    # Laptops
    'airbook 15':'dn-lt-001','airbook 14 8gb':'dn-lt-003','airbook 14':'dn-lt-002',
    'stormbook g9':'dn-lt-010','stormbook g7':'dn-lt-007',
    'stormbook g5 2024':'dn-lt-009','stormbook g5':'dn-lt-008',
    'creatorbook 16':'dn-lt-014','creatorbook 14':'dn-lt-015',
    'flexbook detach':'dn-lt-017','slimbook 14':'nt-lt-001',
    'mini pc m1':'dn-dt-003',
    # Phones
    'x9 pro max':'sf-sp-001','x9 pro':'sf-sp-002','x9 fe':'sf-sp-010','x9':'sf-sp-003',
    'v7 lite':'sf-sp-005','v7':'sf-sp-004',
    # Tablets
    'tab s9 pro':'sf-tb-001','tab s9':'sf-tb-002','tab a5 wifi':'sf-tb-004',
    'tab a5':'sf-tb-003','tab draw pro':'sf-tb-007','tab s8 pro':'sf-tb-008',
    # Headphones
    'headpro x1 se':'ks-hp-002','headpro x1':'ks-hp-001',
    'headon 700':'ks-hp-003','headon 500':'ks-hp-004','headon 300':'ks-hp-005',
    'studiopro m1':'ks-hp-007',
    # Earbuds
    'buds z5 pro gold':'ks-eb-008','buds z5 pro':'ks-eb-001',
    'buds z5':'ks-eb-002','buds z3':'ks-eb-003',
    'buds sport x':'ks-eb-004','buds sport lite':'ks-eb-005',
    'buds z1':'ks-eb-006','novabuds pro':'nt-eb-001',
    # Speakers/Monitors
    'soundpillar 300':'aw-sk-001','proview 27':'aw-mn-001',
    'soundbar pro 500':'ks-sk-001','soundbar 300':'ks-sk-002',
    'boombox x':'ks-sk-003','go mini':'ks-sk-004','homepod one':'ks-sk-006',
    # Accessories
    'hub 7in1':'jc-hb-001','hub 4in1':'jc-hb-002',
    'dock pro':'jc-hb-003','dock airbook':'jc-hb-004',
    'charger 67w':'jc-ch-001','qipad 15':'jc-ch-005','chargepad 15w':'pg-ch-001',
    'powerbank 30000':'pg-pb-002','saifah pen gen 2':'jc-cs-006',
    # Thai names
    'แอร์บุ๊ก 15':'dn-lt-001','แอร์บุ๊ก 14 8gb':'dn-lt-003','แอร์บุ๊ก 14':'dn-lt-002',
    'สตอร์มบุ๊ก g7':'dn-lt-007','สตอร์มบุ๊ก g5 2024':'dn-lt-009','สตอร์มบุ๊ก g5':'dn-lt-008',
    'ครีเอเตอร์บุ๊ก 14':'dn-lt-015',
    'วงโคจร watch s3 ultra':'wk-sw-001','วงโคจร watch s3 pro':'wk-sw-002',
    'วงโคจร watch s3 se':'wk-sw-004','วงโคจร watch s3':'wk-sw-003',
    'แบนด์ 8 pro':'wk-ft-002','แบนด์ 8':'wk-ft-001',
    'เฮดโปร x1 se':'ks-hp-002','เฮดโปร x1':'ks-hp-001',
    'เฮดออน 500':'ks-hp-004','เฮดออน 300':'ks-hp-005',
    'บัดส์ z5 pro':'ks-eb-001','บัดส์ z5':'ks-eb-002',
    'คลื่นเสียง soundbar 300':'ks-sk-002','คลื่นเสียง 300':'ks-sk-002',
    'สายฟ้า x9 pro':'sf-sp-002','สายฟ้า x9':'sf-sp-003',
    'สายฟ้า duopad':'sf-sp-011','แท็บ s8 pro':'sf-tb-008',
    'แท็บ s9 pro':'sf-tb-001','แท็บ a5':'sf-tb-003',
    'สายฟ้า แท็บ a5':'sf-tb-003',
}

def detect_products(query):
    q_lower = query.lower()
    found = {}
    for alias, code in sorted(PRODUCT_ALIASES.items(), key=lambda x: -len(x[0])):
        if alias in q_lower and code not in found.values():
            found[alias] = code
    return list(found.values())


def generate_query_variants(question, choices=None):
    variants = [question]

    filler_re = re.compile(
        r'(ครับ|ค่ะ|คะ|นะ|อ่ะ|ด้วย|เลย|แบบ|ว่า|ที่|ของ'
        r'|ช่วย|ดูให้หน่อย|รบกวน|สอบถาม|อยากทราบ|อยากรู้'
        r'|เฮ้ยย|พอดี|คือ|ว่าไง|ปกติ|จริงๆ|หน่อยนึง)'
    )
    cleaned = filler_re.sub('', question).strip()
    cleaned = re.sub(r'\s+', ' ', cleaned)
    if cleaned != question and len(cleaned) > 10:
        variants.append(cleaned)

    if choices:
        choice_text = ' '.join(
            v for k, v in choices.items()
            if k not in ('9', '10')
            and 'ไม่มีข้อมูล' not in v
            and 'ไม่เกี่ยวข้อง' not in v
            and len(v) < 50
        )
        if choice_text:
            variants.append(question + ' ' + choice_text)

    return variants[:N_QUERIES]


def multi_query_retrieve(question, choices=None, top_k=TOP_K_RETRIEVE):
    variants = generate_query_variants(question, choices)

    all_ranked_lists = []
    for v in variants:
        dense_r = pinecone_retrieve(v, top_k)
        bm25_r  = bm25_retrieve(v, top_k)
        all_ranked_lists.extend([dense_r, bm25_r])

    merged = rrf_merge(all_ranked_lists, top_k=top_k * 2)

    products = detect_products(question)
    if len(products) >= 2:
        per_product = {code: [] for code in products}
        for c in merged:
            for code in products:
                if code in c['source'].lower():
                    per_product[code].append(c); break

        guaranteed, seen = [], set()
        for code in products:
            for c in per_product[code][:4]:
                key = c['source'] + c['text'][:30]
                if key not in seen: seen.add(key); guaranteed.append(c)
        for c in merged:
            if len(guaranteed) >= top_k: break
            key = c['source'] + c['text'][:30]
            if key not in seen: seen.add(key); guaranteed.append(c)
        return guaranteed

    return merged[:top_k]


print('Multi-query retrieval ready')

Multi-query retrieval ready


---
## Section 7: Cross-Encoder Reranker

In [ ]:
from sentence_transformers import CrossEncoder
print('Loading BGE reranker...')
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')
print('Reranker ready')

Loading BGE reranker...


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Reranker ready


In [ ]:
def rerank(query, candidates, top_k=TOP_K_RERANK):
    if not candidates: return []
    pairs  = [(query, c['text'][:512]) for c in candidates]
    scores = reranker.predict(pairs)
    return [c for _, c in sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)[:top_k]]

print('rerank() ready')

rerank() ready


---
## Section 8: LLM — Typhoon API + Choice Shuffling + Majority Vote

In [ ]:
OFF_TOPIC_HARD = [
    'วันหยุดราชการ','ตั๋วเครื่องบิน','ดอกเบี้ยเงินฝาก','ดอกเบี้ยธนาคาร',
    'สูตรอาหาร','ผัดกระเพรา','ต้มยำ','ทำอาหาร','ราคาหุ้น','หุ้นกู้',
    'กองทุนรวม','ดูดวง','หวย','ฟุตบอล','ผลบอล','พยากรณ์อากาศ','จราจร',
]
def is_off_topic(q): return any(kw in q.lower() for kw in OFF_TOPIC_HARD)

BUDGET_RE = re.compile(r'(งบ|มีเงิน|ราคาไม่เกิน|งบไม่เกิน)\s*[฿]?\s*([\d,]+)')

def get_top_k(question):
    if BUDGET_RE.search(question) or 'กี่รุ่น' in question: return 40, 15
    if len(detect_products(question)) >= 2: return 35, 12
    if any(kw in question for kw in ['ค่าส่','กี่วัน','จัดส่ง','เกาะ']): return 25, 8
    if len(question) > 100: return 30, 12
    return 25, 10


SYSTEM_PROMPT = """คุณคือผู้ช่วยตอบคำถามเกี่ยวกับร้านอิเล็กทรอนิกส์ "ฟ้าใหม่" (FahMai)
คุณได้รับข้อมูลอ้างอิง (context) และคำถามพร้อมตัวเลือก

กฎการตอบ:
1. อ่าน context อย่างละเอียด
2. ตรวจสอบ *ทุก* ตัวเลือก 1-8 เปรียบเทียบกับ context ก่อนตัดสิน
3. เลือกตัวเลือกที่เนื้อหาตรงกับ context มากที่สุด
4. ข้อ 9 = context ไม่มีข้อมูลนั้นเลย (ไม่ใช่คำถามยาก)
5. ข้อ 10 = คำถามไม่เกี่ยวกับร้านฟ้าใหม่เลย
6. ตอบด้วยตัวเลขเดียว ห้ามอธิบาย

คำเตือน: คำตอบที่ถูกต้องอาจอยู่ตัวเลือกท้ายรายการ อย่าเลือกตัวเลือกแรกที่ดูใกล้เคียง"""

FEW_SHOT = [
    {
        'role': 'user',
        'content': """Context:\n[หมวด: products] [สินค้า: วงโคจร Watch S3 Ultra]\nกันน้ำ: 10 ATM\n\n=== คำถาม ===\nWatch S3 Ultra กันน้ำได้กี่ ATM?\n=== ตัวเลือก ===\n1. IP67  2. 1 ATM  3. 3 ATM  4. IP68  5. IPX8  6. 20 ATM  7. 5 ATM  8. 10 ATM\n9. ไม่มีข้อมูล  10. ไม่เกี่ยวข้อง"""
    },
    {'role': 'assistant', 'content': '8'},
    {
        'role': 'user',
        'content': """Context:\n[หมวด: products] [สินค้า: สายฟ้า Phone X9]\nราคา: ฿29,990\n\n=== คำถาม ===\nสายฟ้า Phone X9 Pro Max ผลิตที่ประเทศอะไร?\n=== ตัวเลือก ===\n1. เกาหลีใต้  2. สหรัฐ  3. จีน  4. ญี่ปุ่น  5. ไทย  6. อินเดีย  7. เวียดนาม  8. ไต้หวัน\n9. ไม่มีข้อมูล  10. ไม่เกี่ยวข้อง"""
    },
    {'role': 'assistant', 'content': '9'},
    {
        'role': 'user',
        'content': """Context:\n[หมวด: policies] คืนสินค้าภายใน 7 วัน\n\n=== คำถาม ===\nราคาหุ้น Apple?\n=== ตัวเลือก ===\n1. 200 USD  2. 150 USD  3. 130 USD  4. 220 USD  5. 180 USD  6. 100 USD  7. 250 USD  8. 160 USD\n9. ไม่มีข้อมูล  10. ไม่เกี่ยวข้อง"""
    },
    {'role': 'assistant', 'content': '10'},
]

print('Prompts ready')

Prompts ready


In [ ]:
def ask_llm(messages, max_retries=5):
    """Call Typhoon API (thaillm.or.th)."""
    url     = 'http://thaillm.or.th/api/typhoon/v1/chat/completions'
    headers = {'Content-Type': 'application/json', 'apikey': THAILLM_API_KEY}
    payload = {
        'model':       '/model',
        'messages':    messages,
        'max_tokens':  8,
        'temperature': 0.0,
        'top_p':       1.0,
    }
    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)
            if resp.status_code == 429:
                wait = min(2**attempt, 30)
                print(f'  rate limit {wait}s...'); time.sleep(wait); continue
            resp.raise_for_status()
            return resp.json()['choices'][0]['message']['content'].strip()
        except requests.exceptions.RequestException as e:
            wait = min(2**attempt, 30)
            print(f'  error: {e}, retry {wait}s...'); time.sleep(wait)
        except Exception as e:
            print(f'  parse error: {e}'); return None
    return None


def parse_answer(text):
    if text is None: return None
    clean = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL|re.IGNORECASE)
    clean = re.sub(r'<think>.*', '', clean, flags=re.DOTALL|re.IGNORECASE).strip()
    m = re.match(r'^(\d{1,2})\s*$', clean)
    if m and 1 <= int(m.group(1)) <= 10: return int(m.group(1))
    m = re.search(r'ANSWER[:\s]*(\d{1,2})', clean, re.IGNORECASE)
    if m and 1 <= int(m.group(1)) <= 10: return int(m.group(1))
    for pat in [r'คำตอบ[:\s]*(\d+)', r'ข้อ\s*(\d+)', r'ตอบ\s*(\d+)']:
        m = re.search(pat, clean)
        if m and 1 <= int(m.group(1)) <= 10: return int(m.group(1))
    for n in reversed(re.findall(r'\b(\d{1,2})\b', clean)):
        if 1 <= int(n) <= 10: return int(n)
    return None


def shuffle_choices(choices, seed):
    rng = random.Random(seed)
    keys_1_8 = [str(i) for i in range(1, 9)]
    shuffled_keys = keys_1_8[:]
    rng.shuffle(shuffled_keys)

    new_choices, reverse_map = {}, {}
    for new_pos, orig_key in enumerate(shuffled_keys, 1):
        new_choices[str(new_pos)] = choices[orig_key]
        reverse_map[new_pos] = int(orig_key)
    new_choices['9']  = choices['9']
    new_choices['10'] = choices['10']
    reverse_map[9]  = 9
    reverse_map[10] = 10
    return new_choices, reverse_map


def format_context(chunks, max_chunks=15):
    return '\n\n'.join(
        f"=== ข้อมูลที่ {i} (จาก: {c.get('source','?')}) ===\n{c['text']}"
        for i, c in enumerate(chunks[:max_chunks], 1)
    )

def format_question(question, choices):
    lines = [f'\n=== คำถาม ===\n{question}\n=== ตัวเลือก ===']
    for k in sorted(choices.keys(), key=int):
        lines.append(f'{k}. {choices[k]}')
    return '\n'.join(lines)


def ask_one_vote(context_str, question, choices, seed):
    shuffled, reverse_map = shuffle_choices(choices, seed)
    user_msg = f'Context:\n{context_str}\n{format_question(question, shuffled)}'
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        *FEW_SHOT,
        {'role': 'user', 'content': user_msg},
    ]
    raw = ask_llm(messages)
    shuffled_ans = parse_answer(raw)
    if shuffled_ans is None: return None, raw
    return reverse_map.get(shuffled_ans, shuffled_ans), raw


print('LLM + shuffling functions ready')

LLM + shuffling functions ready


---
## Section 9: Answer Pipeline

In [ ]:
def answer_question(q, verbose=False):
    """
    Pinecone v5 pipeline:
    1. Off-topic heuristic → 10
    2. Multi-query retrieval (3 variants × Pinecone dense + BM25 → RRF)
    3. Cross-encoder rerank
    4. Shipping policy boost
    5. Choice shuffling × N_VOTES → majority vote
    """
    query   = q['question']
    choices = q['choices']

    # 1. Off-topic
    if is_off_topic(query):
        if verbose: print('  off-topic → 10')
        return 10, [10]

    # 2. Multi-query retrieval
    retrieve_k, rerank_k = get_top_k(query)
    candidates = multi_query_retrieve(query, choices=choices, top_k=retrieve_k)

    # 3. Rerank
    top_chunks = rerank(query, candidates, top_k=rerank_k)

    # 4. Shipping boost
    if any(kw in query for kw in ['ค่าส่','กี่วัน','จัดส่ง','เกาะ','ระยะเวลา']):
        ship = [c for c in candidates if 'shipping' in c['source'].lower()]
        rest = [c for c in top_chunks  if 'shipping' not in c['source'].lower()]
        top_chunks = (ship + rest)[:rerank_k]

    context_str = format_context(top_chunks)

    # 5. Shuffled voting
    votes = []
    for vote_idx in range(N_VOTES):
        seed = vote_idx * 137
        ans, raw = ask_one_vote(context_str, query, choices, seed)
        if ans is not None:
            votes.append(ans)
        if verbose:
            print(f'  vote {vote_idx} (seed={seed}): raw="{str(raw)[:30]}" → {ans}')
        time.sleep(0.15)

    # Majority vote
    if not votes:
        final = 9
    else:
        counts  = Counter(votes)
        max_cnt = max(counts.values())
        winners = [k for k, v in counts.items() if v == max_cnt]
        final   = winners[0]

    if verbose:
        print(f'  votes={votes} → final={final}')
        print(f'  sources: {[c["source"] for c in top_chunks[:4]]}')

    return final, votes


print('answer_question() ready')

answer_question() ready


---
## Section 10: Full Run + Submission

In [ ]:
questions = []
with open(f'{DATA_DIR}/questions.csv', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        choices = {str(i): row[f'choice_{i}'] for i in range(1, 11)}
        questions.append({'id': int(row['id']), 'question': row['question'], 'choices': choices})
print(f'Loaded {len(questions)} questions')

Loaded 100 questions


In [ ]:
results, errors = [], []

for q in questions[:N_QUESTIONS]:
    print(f"Q{q['id']:03d}: {q['question'][:50]}...", end='  ')
    try:
        ans, _ = answer_question(q, verbose=False)
        if ans is None:
            ans = 9
            errors.append(q['id'])
        results.append({'id': q['id'], 'answer': ans})
        print(f'→ {ans}')
    except Exception as e:
        print(f'ERROR: {e}')
        results.append({'id': q['id'], 'answer': 9})
        errors.append(q['id'])

print(f'\nDone: {len(results)}/100, fallbacks={len(errors)}')


Q001: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ...  

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

→ 5
Q002: ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ...  

...

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

→ 4
Q100: สตอร์มบุ๊ก G5 ราคา ฿27,990 ซื้อมาได้ 5 วัน ยังไม่แ...  

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

→ 3

Done: 100/100, fallbacks=0


In [ ]:
import pandas as pd

submission = pd.DataFrame(results).sort_values('id')
missing = {q['id'] for q in questions} - set(submission['id'])
if missing:
    submission = pd.concat([
        submission,
        pd.DataFrame({'id': list(missing), 'answer': 9})
    ]).sort_values('id')

out_path = './submission_v5_pinecone_4015.csv'
submission.to_csv(out_path, index=False)
print(f'Saved {len(submission)} rows → {out_path}')
print(submission['answer'].value_counts().sort_index())

Saved 100 rows → ./submission_v5_pinecone_4015.csv
answer
1     13
2     12
3     15
4     12
5      7
6     13
7     10
8      5
9      8
10     5
Name: count, dtype: int64
